# Shock Tube Problem 101
**강좌**: *전산유체역학*

## 충격파관 문제 2차원 확장
- 관 가운데 격막을 놓고, 좌우를 다르게 충진시킨 후 격막을 터트린다. 초기 조건에 따라 다양한 충격파-팽창파 구조가 발달한다.

- 계산 영역을 y방향으로 확장한 2차원 영역 $[-0.5, 0.5]\times[0, 0.1]$로 한다.

    - `grid` 폴더 내 `tube.xyz` 파일을 사용한다.

### SOD shock tube problem
팽창파 - 미끄럼유선 - 충격파가 발달하는 충격파관 문제이다.

$$
(\rho, u, v, p) 
= \begin{cases}
(1.0, 0,0 0.0, 1.0) & x < 0.0 \\
(0.125, 0.0, 0.0, 0.1) & x \ge 0.0
\end{cases}
$$

In [1]:
%matplotlib inline
from matplotlib import pylab as plt

import numpy as np

plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 150

## 구현
## 공간차분항 (RHS) 계산
- Transformed governing equation에서 시작한다.

    $$
    {\frac{\mathbf{U}_t}{J}} + \tilde{\mathbf{F}}_{\xi} + \tilde{\mathbf{G}}_{\eta} = 0.
    $$

    - 보존항 벡터를 저장하므로 $\tilde{\mathbf{U}}$ 가 아닌 $\mathbf{U}/J$ 로 계산한다.
    
- Semi discrtet form

    $$
    \frac{d\mathbf{U}_{i,j}}{dt} = \mathbf{R}_{i,j} = -J_{i,j} (\tilde{\mathbf{F}}_{i+1/2,j} - \tilde{\mathbf{F}}_{i-1/2,j} + \tilde{\mathbf{G}}_{i,j+1/2} - \tilde{\mathbf{G}}_{i,j-1/2})
    $$

In [2]:
from nbutils import array

import numba as nb
import numpy as np


def make_rhside(nfx, nfy, jac, si, sj, flux, bc, nvars=4, npads=(1,1)):          
    """
    Kernal generator for right hand side (First order)
    
    Parameters
    ----------
    nfx : integer
        xi 방향 격자 개수
    nfy : integer
        eta 방향 격자 개수
    jac : array
        Jacobian 크기
    si : array
        xi 방향 Face metric
    sj : array
        eta 방향 Face metric
    flux : function
        근사 리만 해석자 함수
    bc : function
        경계조건 함수
    nvars : integer
        벡터 크기 (기본값: 4)
    npad : integer
        경계를 위한 padding 개수 (기본값: (1,1))
    """
    @nb.jit(nopython=True)
    def _run(u, du, ff, fg):
        # Allocate local static array
        fn = array((nvars,))
        
        # Compute BC
        bc(u)
        
        # xi sweep
        for j in range(nfy):
            jy = j + npads[1]
            for i in range(nfx+1):
                ix = i + npads[0]
                
                ul = u[:, jy, ix-1]
                ur = u[:, jy, ix]
                n = si[:, j, i]
                flux(ul, ur, n, fn)
                ff[:, j, i] = fn
                
        # Eta sweep
        for j in range(nfy+1):
            jy = j + npads[1]
            for i in range(nfx):
                ix = i + npads[0]
                
                ul = u[:, jy-1, ix]
                ur = u[:, jy, ix]
                n = sj[:, j, i]
                flux(ul, ur, n, fn)
                fg[:, j, i] = fn

        # negative derivative of flux at each cell
        for j in range(nfy):
            jp = j + 1
            jy = j + npads[1]
            for i in range(nfx):        
                ip = i + 1
                ix = i + npads[0]
                
                du[:, jy, ix] = -jac[j,i]*(
                    ff[:, j, ip] - ff[:,j, i] 
                    + fg[:, jp, i] - fg[:, j, i]
                )
            
    return _run

- Riemann 문제 해석
    * 수치적으로 근사 Riemann 해석을 위한 Flux 기법을 적용한다.   
    
        $$
        \tilde{\mathbf{F}}_{i+1/2,j} = \mathbf{H}(\mathbf{U}_{i,j}, \mathbf{U}_{i+1,j}, \mathbf{n}), 
        \tilde{\mathbf{G}}_{i,j+1/2} = \mathbf{H}(\mathbf{U}_{i,j}, \mathbf{U}_{i,j+1}, \mathbf{n})
        $$
  
        * $n$ 은 Face metric인 $|\nabla \xi| / J$ 또는 $|\nabla \eta| / J$을 사용한다.
    
    * Rusanov flux: Spectral radius를 이용하는 가장 간단한 Flux 기법이다.
    
       $$
       \mathbf{H}(\mathbf{U}_L, \mathbf{U}_R, \mathbf{n}) = \frac{\mathbf{F}_{n,L} + \mathbf{F}_{n,R}}{2} - \frac{|U_{n,j+1/2}| + a_{j+1/2}}{2} (\mathbf{U}_R - \mathbf{U}_L)
       $$

In [3]:
%%writefile solvers/rsolvers/rusanov.py
from nbutils import array
from solvers.fluid import to_flux

import numba as nb
import numpy as np


def make_rusanov(gamma=1.4, nvars=4):
    """
    Rusanov flux 계산 함수 생성
    
    Parameters
    ----------
    gamma : float
        Specific heat ratio (기본값: 1.4)
    nvars : integer
        벡터 크기 (기본값: 4)
        
    Return
    ------
    _flux : function
        Rusanov flux 함수
    """
    
    @nb.jit(nopython=True)
    def _flux(ul, ur, n, fn):
        fl, fr = array((nvars,)), array((nvars,))
        
        pl, vl = to_flux(ul, fl, n)
        pr, vr = to_flux(ur, fr, n)
        
        vn = 0.5*(vl + vr)
        an = np.sqrt(gamma*(pl+pr) / (ul[0] + ur[0]))*np.sqrt(n[0]**2 + n[1]**2) + np.abs(vn)
        
        for jdx in range(nvars):
            fn[jdx] = 0.5*(fl[jdx] + fr[jdx]) - 0.5*an*(ur[jdx] - ul[jdx])
            
    return _flux

Overwriting solvers/rsolvers/rusanov.py


- 경계조건
    - 경계는 imin, imax, jmin, jmax 값에 따른 Line에 대해 계산해야 한다.

    - 각 경계선을 따라서 $n_{pad}$ 개의 행을 추가한다.
        - 계산 영역 Index (Zero-based numbering) : $u[n_{pad}:n_{fy} + n_{pad},n_{pad}:n_{fx} + n_{pad}]$
  
    - imin, imax 면에서는 Zero Gradient 경계조건을 부여
    
      $$
      \begin{align}
      u[j,n_{pad}-1] &= u[j,n_{pad}],&j=n_{pad},...,n_{fy}+n_{pad}-1\\
      u[j,n_{fx} + n_{pad}] &= u[j,n_{fx} + n_{pad}-1],&j=n_{pad},...,n_{fy}+n_{pad}-1 \\
      \end{align}
      $$
 
    - jmin, jmax 면에서는 대칭 (Periodic) 경계조건을 부여
 
      $$
      \begin{align}
      u[n_{pad}-1,i] &= u[n_{fy} + n_{pad}-1,i],&i=n_{pad},...,n_{fx}+n_{pad}-1\\
      u[n_{fy} + n_{pad},i] &= u[n_{pad},i],&i=n_{pad},...,n_{fx}+n_{pad}-1
      \end{align}
      $$ 

In [4]:
%%writefile solvers/bcs/zero.py
import numba as nb


def make_bc_zero(nvars=4, **kwargs):
    """
    Zero gradient BC function generator

    Parameter
    ---------
    nvars : integer
        변수 개수

    Return
    ------
    _bc : jitted function
        Zero BC function
    """
    @nb.jit(nopython=True)
    def _bc(ul, ur, nf):
        for k in range(nvars):
            ur[k] = ul[k]
            
    return _bc

Overwriting solvers/bcs/zero.py


In [5]:
from nbutils import array
from solvers.bcs.zero import make_bc_zero

import numba as nb
import numpy as np


def make_bc(nfx, nfy, si, sj, nvars=4, npads=(1,1)):
    """
    BC kernel genertor (First version)

    Parameter
    ---------
    nfx - integer
        xi 방향 격자 개수
    nfy - integer
        eta 방향 격자 개수
    si : array
        xi 방향 Face metric
    sj : array
        eta 방향 Face metric
    nvars : integer
        벡터 크기 (기본값: 4)
    npads : integer
        경계를 위한 padding 개수 (기본값: (1,1))
    """
    # Call BC fucntion
    # TODO: will be generalized
    bcf = make_bc_zero(nvars=nvars)
    
    # Compute unit normal vector
    ni = si / np.linalg.norm(si, axis=0)
    nj = sj / np.linalg.norm(sj, axis=0) 
    
    # TODO: will be generalized
    # imin / imax : zero gradient BC
    bc_imin = make_bc_i(npads[0], npads[0]-1, nfy, bcf, ni, nvars, npads)
    bc_imax = make_bc_i(nfx+npads[0]-1, nfx+npads[0], nfy, bcf, ni, nvars, npads)

    # jmin / jmax : periodic BC
    bc_jmin = make_bc_j_periodic(npads[1], nfy+npads[1], nfx, nvars, npads[0])
    bc_jmax = make_bc_j_periodic(nfy+npads[1] - 1, npads[1]-1, nfx, nvars, npads[0])
    
    @nb.jit(nopython=True)
    def _run(u):
        bc_imin(u)
        bc_imax(u)
        bc_jmin(u)
        bc_jmax(u)
        
    return _run


def make_bc_i(i, ip, nfy, bcf, nf, nvars=4, npads=(1,1)):
    """
    i 방향 BC Sweep 커널 생성

    Parameter
    ---------
    i - integer
        Innder index
    ip - integer
        Ghost cell index
    nfy - integer
        eta 방향 격자 개수
    bcf - function
        경계조건 함수
    nf - array
        수직벡터
    nvars : integer
        벡터 크기 (기본값: 4)
    npads : integer
        경계를 위한 padding 개수 (기본값: (1,1))
    """
    @nb.jit(nopython=True)
    def _run(u):               
        ur = array((nvars,))
        
        for j in range(npads[1], npads[1]+nfy):
            bcf(u[:, j, i], ur, nf[:, j-npads[1], i-npads[0]])
            for k in range(nvars):
                u[k, j, ip] = ur[k]
                
    return _run


def make_bc_j_periodic(j, jp, nfx, nvars=4, npad=1):
    """
    j 방향 대칭 BC 커널 생성

    Parameter
    ---------
    j - integer
        Innder index
    jp - integer
        Ghost cell index
    nfx - integer
        xi 방향 격자 개수
    nvars : integer
        벡터 크기 (기본값: 4)
    npad : integer
        j 방향 경계를 위한 padding 개수 (기본값: 1)
    """
    @nb.jit(nopython=True)
    def _run(u):
        for k in range(nvars):
            for i in range(npad, npad+nfx):
                u[k, jp, i] = u[k, j, i]
                
    return _run

## 시간 적분
### 시간간격 결정
- CFL 조건을 이용해서 매 시간 해를 계산한다.

$$
\Delta t  = CFL \frac{1}{(V \cdot \nabla \xi + a |\nabla \xi|+ V \cdot \nabla \eta + a |\nabla \eta|) }
$$

- `make_timestep` 함수에서 시간 간격을 계산하는 kernel을 생성한다.

In [6]:
%%writefile solvers/unsteady/timestep.py
from nbutils import array
from solvers.fluid import to_primitive

import numba as nb
import numpy as np


def make_timestep(nfx, nfy, jac, si, sj, cfl, gamma=1.4, nvars=4, npads=(1.1)):
    """
    Time step calulcation kernel generator
    
    Parameters
    ----------
    nfx - integer
        xi 방향 격자 개수
    nfy - integer
        eta 방향 격자 개수
    jac : array
        Jacobian 크기
    si : array
        xi 방향 Face metric
    sj : array
        eta 방향 Face metric
    cfl : float
        CFL 값
    gamma : float
        비열비 (기본값 1.4)
    nvars: integer
        벡터 크기
    npad : integer
        Solution 벡터의 Padding 개수
        
    Returns
    -------
    _timestep : kernal
        시간 간격 계산 Kerenl
    """
    # Metric at the cell
    sic = 0.5*(si[:, :, :-1] + si[:, :, 1:])*jac
    sjc = 0.5*(sj[:, :-1, :] + sj[:, 1:, :])*jac

    # Magnitude of the metric
    aic = np.linalg.norm(sic, axis=0)
    ajc = np.linalg.norm(sjc, axis=0)
        
    @nb.jit(nopython=True)
    def _timestep(q):
        # Allocate local static array
        qp = array((nvars,))
        
        dtmin = 1e8
        
        for j in range(nfy):
            jy = j + npads[1]
            for i in range(nfx):
                ix = i + npads[0]
                
                to_primitive(q[:, jy, ix], qp)
                rho, u, v, p = qp
                a = np.sqrt(gamma*p/rho)

                ui = sic[0, j, i]*u + sic[1, j, i]*v
                vj = sjc[0, j, i]*u + sjc[1, j, i]*v

                dt = cfl  / ((abs(ui) + a*aic[j, i]) + (abs(vj) + a*ajc[j, i]))
                dtmin = min(dt, dtmin)

        return dtmin
    
    return _timestep

Overwriting solvers/unsteady/timestep.py


### Explicit Euler 기법

$$
\mathbf{U}^{n+1}_j = \mathbf{U}^n_j + \Delta t \mathbf{R}_j
$$

In [7]:
%%writefile solvers/unsteady/expliciteuler.py
import numpy as np


def make_expliciteuler(nfx, nfy, rhside, nvars=4, npads=(1,1)):
    """
    Explicit Euler 계산 커널 생성
    
    Parameters
    ----------
    nfx - integer
        xi 방향 격자 개수
    nfy - integer
        eta 방향 격자 개수
    rhside : kernel
        우변 계산 커널
    nvars : integer
        벡터 크기 (기본값: 3)
    npad : integer
        Solution 벡터의 Padding 개수
    """
    # Temporal arrays
    du = np.zeros((nvars, nfy+2*npads[1], nfx+2*npads[0]))
    
    # Temporal arrays for flux
    ff = np.zeros((nvars, nfy, nfx+1))
    fg = np.zeros((nvars, nfy+1, nfx))
    
    def _step(dt, u):
        rhside(u, du, ff, fg)
        u += dt*du

    return _step

Overwriting solvers/unsteady/expliciteuler.py


## 실행
- 격자 생성
- 계산 벡터 생성 : $\mathbf{U}$
- Kernels 생성
- 초기화
- Main Loop

In [8]:
from matplotlib import pylab as plt

# 모듈 로드
from solvers.plot3d import read_plot3d_b, write_q
from solvers.grid import cell_jacobian, face_metric, cell_x
from solvers.fluid import to_conservative, to_primitive
from solvers.rsolvers.rusanov import make_rusanov
from solvers.unsteady.expliciteuler import make_expliciteuler
from solvers.unsteady.timestep import make_timestep

import numpy as np

plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 150

In [9]:
# Constants
gamma = 1.4
nvars = 4
npads = (1,1)

cfl = 0.9

# Initial conditions (Primitive)
qpl = (1.0, 0.0, 0.0, 1.0)
qpr = (0.125, 0.0, 0.0, 0.1)
xbk = 0.0
target_time = 0.2

# Plot3D 격자 읽기
raw = read_plot3d_b('grids/tube.xyz')

# Parse point
x, y = raw[:2, 0]

# Compute area and metric
jac = cell_jacobian(x ,y)
si, sj = face_metric(x, y)

# Cell Center
xc, yc =cell_x(x, y)

# Array
jmx, imx = x.shape
jmx -= 1
imx -= 1
q = np.empty((nvars, jmx+2*npads[1], imx+2*npads[0]))

# Generate kernels
bc = make_bc(imx, jmx, si, sj, nvars, npads)
flux = make_rusanov(gamma, nvars)
rhside = make_rhside(imx, jmx, jac, si, sj, flux, bc, nvars, npads)
timestep = make_timestep(imx, jmx, jac, si, sj, cfl, gamma, nvars, npads)
step = make_expliciteuler(imx, jmx, rhside, nvars, npads)

# Initialize
ql = np.empty(nvars)
qr = np.empty(nvars)

to_conservative(qpl, ql)
to_conservative(qpr, qr)

for j in range(jmx):
    for i in range(imx):
        if xc[j,i] < xbk:
            q[:, j+npads[1], i+npads[0]] = ql
        else:
            q[:, j+npads[1], i+npads[0]] = qr

# Main Loop        
t = 0
n = 0
while abs(t - target_time) > np.finfo(1.0).eps:
    # Compute time step
    dt = min(timestep(q), target_time - t)
    
    # Step
    step(dt, q)
    t += dt
    n += 1

print("{} iterations until t={:.3f}".format(n, t))

150 iterations until t=0.200


In [10]:
# Write solution
bc(q)

# Adjust solution array considering padding
_, jqy, iqx = q.shape
ie = (0, iqx)
je = (0, jqy)

if npads[0] > 1:
    ie = npads[0] - 1, -(npads[0] - 1)
    
if npads[1] > 1:
    je = npads[1] - 1, -(npads[1] - 1)

write_q('sol.q', q[:, je[0]:je[1], ie[0]:ie[1]])

## 가시화
- Paraview 또는 Tecplot으로 유동을 확인하다.

    ```bash
    user@localhost: ~$ paraview grids/tube.xyz
    ```

    - `sol.q` 파일을 Q File Name으로 설정한다.